In [2]:
import pandas as pd
from pycaret.regression import *

In [3]:
df = pd.read_csv('/content/tripadvisor_hotel_reviews.csv')
df

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5
...,...,...
20486,"best kept secret 3rd time staying charm, not 5...",5
20487,great location price view hotel great quick pl...,4
20488,"ok just looks nice modern outside, desk staff ...",2
20489,hotel theft ruined vacation hotel opened sept ...,1


In [4]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from string import punctuation
import unicodedata

def clean(text):
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    text = re.sub(r'-?\b\d+(\.\d+)?\b|-?\b\.\d+\b', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b[^a-zA-Z\s]+\b', ' ', text)
    text = re.sub(r'\b(\w)\1{2,}\w*\b', ' ', text)
    text = re.sub(r'\b(?=\w*\d)(?=\w*[a-zA-Z])\w+\b', '', text)
    text = text.lower()
    punc = list(punctuation)
    tokens = word_tokenize(text)
    stop = list(stopwords.words("english"))+punc
    words = [words for words in tokens if words not in stop]
    lemm = WordNetLemmatizer()
    cleaned = [lemm.lemmatize(word) for word in words]
    return ' '.join(cleaned)


In [6]:
df['clean_rev'] = df['Review'].apply(clean)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfid_ng = TfidfVectorizer(
    ngram_range=(1,3),
    lowercase=True,
)
X = tfid_ng.fit_transform(df['clean_rev'])
voc = tfid_ng.get_feature_names_out(df['clean_rev'])

In [10]:
y = df['Rating']

In [11]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_smote,y_smote = sm.fit_resample(X,y)

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_smote,y_smote,random_state=42,train_size=0.2)

In [13]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

results = []


models = {
    "Linear Regression" :LinearRegression(),
    'Ridge':Ridge(),
    'LassoRegression':Lasso(),
    'KNeighborsRegressor' : KNeighborsRegressor()

}
for model in models.values():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results.append({
        'Model': model.__class__.__name__,
        'MSE': mse,
        'R2': r2,
        'MAE': mae,
        'MAPE': mape
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)

                 Model       MSE        R2       MAE      MAPE
0     LinearRegression  0.303903  0.847553  0.324909  0.098392
1                Ridge  0.454956  0.771781  0.515007  0.224427
2                Lasso  1.993609 -0.000051  1.198479  0.626127
3  KNeighborsRegressor  1.705810  0.144317  0.898089  0.254202


In [24]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score,mean_absolute_error,mean_absolute_percentage_error,mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV



models = {
    "Linear Regression (grid)" :GridSearchCV(LinearRegression(),param_grid={'fit_intercept':[True,False]}),
}

for model in models.values():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    results.append({
        'Model': model.__class__.__name__,
        'MSE': mse,
        'R2': r2,
        'MAE': mae,
        'MAPE': mape
    })

metrics_df = pd.DataFrame(results)
print(metrics_df)

                 Model       MSE        R2       MAE      MAPE
0     LinearRegression  0.303903  0.847553  0.324909  0.098392
1                Ridge  0.454956  0.771781  0.515007  0.224427
2                Lasso  1.993609 -0.000051  1.198479  0.626127
3  KNeighborsRegressor  1.705810  0.144317  0.898089  0.254202
4         GridSearchCV  0.303903  0.847553  0.324909  0.098392


In [21]:
import pickle
pickle.dump(models['LinearRegression'],open('lr.pkl','wb'))